# Module `validators.py`

Ce notebook detaille toutes les validations appliquees au projet. Elles sont separees du generateur pour que chaque brique reste responsable d'une seule chose.

Le module expose :

- des helpers bas niveau (`density_from_edge_count`, `average_degree_from_edge_count`, `density_in_bounds`, `is_connected`) ;
- `InstanceValidator` pour valider une instance statique ;
- `DynamicStateValidator` pour valider un etat dynamique avant de couper une arete.

La philosophie est defensive : les tirages aleatoires peuvent proposer des graphes ou des etats absurdes, mais les validateurs empechent ces candidats d'entrer dans le reste du pipeline.


In [ ]:
import sys
from pathlib import Path
from pprint import pprint

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_network import DynamicNetworkSimulator
from cesipath.graph_generator import GraphGenerator
from cesipath.models import EdgeStatus, GraphGenerationConfig
from cesipath.validators import (
    DynamicStateValidator,
    InstanceValidator,
    average_degree_from_edge_count,
    density_from_edge_count,
    is_connected,
)

## 1. Helpers bas niveau

Trois fonctions pures servent de briques a tout le reste :

- `density_from_edge_count(n, m) = m / (n*(n-1)/2)` : densite d'un graphe non oriente simple [1] ;
- `average_degree_from_edge_count(n, m) = 2m / n` : degre moyen non oriente, consequence du lemme des poignees de main [1] ;
- `is_connected(n, edges)` : parcours BFS depuis le sommet 0 pour verifier que tous les sommets sont atteignables [2].

Ces helpers sont volontairement independants de `GraphInstance`. Ils peuvent donc etre testes avec de petits graphes artificiels, comme une chaine ou deux composantes separees.

**Pourquoi la connexite ne suffit pas ?** Une chaine de `n` sommets est connexe, mais elle a un degre moyen proche de 2 et chaque arete interne est critique. Pour un reseau routier dynamique, on veut aussi une densite et un degre moyen minimaux.


In [ ]:
n = 6
edges = {(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)}  # chaine
print("Densite   :", round(density_from_edge_count(n, len(edges)), 3))
print("Degre moy :", round(average_degree_from_edge_count(n, len(edges)), 3))
print("Connexe ? :", is_connected(n, edges))

edges_broken = {(0, 1), (2, 3)}
print("Connexe (casse) ?", is_connected(n, edges_broken))

## 2. `InstanceValidator` - 4 criteres statiques

Appelee par `GraphGenerator.generate()` dans la boucle de rejet. Une instance est valide **si et seulement si** tous les criteres ci-dessous passent :

| # | Critere | Source |
|---|---|---|
| 1 | `base_density` dans `[resolved_min_base_density, resolved_max_base_density]` | configuration |
| 2 | `residual_density` dans `[resolved_min_residual_density, resolved_max_residual_density]` | configuration |
| 3 | `residual_average_degree >= resolved_min_average_residual_degree` | configuration derivee |
| 4 | graphe residuel connexe | BFS sur aretes non `FORBIDDEN` [2] |

**Pourquoi valider apres generation ?** Parce que certaines proprietes dependent du hasard. Le generateur protege un arbre couvrant, mais les tirages de densification, surcharge et interdiction peuvent quand meme produire une instance trop dense, trop pauvre ou peu interessante. La boucle de rejet permet de jeter ces candidats et de recommencer proprement.

Le validateur ne corrige pas l'instance. Il dit seulement oui/non. Cette simplicite est volontaire : la correction automatique melangerait generation et validation, et rendrait le comportement plus difficile a expliquer.


In [ ]:
config = GraphGenerationConfig(node_count=8, seed=42)
instance = GraphGenerator(config).generate()
validator = InstanceValidator(config)

print("Instance valide ?", validator.is_valid(instance))
print()
print("  base_density        :", round(instance.base_density, 4),
      "in [", config.resolved_min_base_density, ",", config.resolved_max_base_density, "]")
print("  residual_density    :", round(instance.residual_density, 4),
      "in [", config.resolved_min_residual_density, ",", config.resolved_max_residual_density, "]")
print("  residual_avg_degree :", round(instance.residual_average_degree, 4),
      ">=", config.resolved_min_average_residual_degree)
print("  residuel connexe    :", InstanceValidator.is_residual_graph_connected(instance))

## 3. `DynamicStateValidator` - 4 invariants dynamiques

Appelee par `DynamicNetworkSimulator.advance()` **avant** chaque coupure d'arete. Si l'un des invariants casserait, la coupure est refusee et l'arete reste active.

| # | Invariant | Source |
|---|---|---|
| 1 | graphe actif connexe | `is_connected` |
| 2 | densite active suffisante | `resolved_dynamic_min_density` |
| 3 | degre moyen actif suffisant | `resolved_dynamic_min_average_degree` |
| 4 | ratio OFF pas trop eleve | `dynamic_max_disabled_ratio` |

Le validateur travaille sur un ensemble d'aretes actives, pas sur la matrice complete, parce que connexite et degre sont des proprietes du graphe physique [3]. C'est logique : une matrice complete peut rester lisible meme si elle encode des distances infinies ou des chemins impossibles. Les invariants doivent etre verifies au niveau du reseau physique.

**Detail important** : `can_disable_edge` teste le monde apres retrait de l'arete candidate. On ne coupe jamais d'abord pour verifier ensuite ; cela evite de produire temporairement un snapshot invalide.


In [ ]:
simulator = DynamicNetworkSimulator(instance, seed=42)
snapshot = simulator.initialize_snapshot()
dynamic_validator = DynamicStateValidator(instance)

candidate = next(iter(snapshot.edge_availability))
print("Arete candidate :", candidate)
print("Peut-on la passer OFF ?", dynamic_validator.can_disable_edge(snapshot.edge_availability, candidate))

## 4. Contre-exemple : casser la connexite

Pour illustrer la robustesse, on cree manuellement un graphe en chaine `0-1-2-3`, on met toutes les aretes sur `ON`, puis on tente de couper l'arete centrale `(1, 2)`. Cela separerait le graphe en deux composantes : `{0, 1}` et `{2, 3}`.

Ce cas est volontairement minimal. Il montre pourquoi la validation dynamique doit raisonner sur les aretes actives avant d'accepter une coupure. Sans ce test, Dijkstra trouverait des distances infinies entre certaines paires, et les solveurs recevraient une matrice qui ne correspond plus a un probleme VRP exploitable.


In [ ]:
# On construit un petit instance artificielle via l'instance reelle puis on forge les aretes.
from dataclasses import replace

config_chain = GraphGenerationConfig(node_count=4, seed=1, auto_density_profile=False,
                                     min_base_density=0.0, max_base_density=1.0,
                                     min_residual_density=0.0, max_residual_density=1.0,
                                     min_average_residual_degree=0.0,
                                     dynamic_max_disabled_ratio=0.5)
instance_chain = GraphGenerator(config_chain).generate()

# On force une configuration en chaine : seules (0,1), (1,2), (2,3) sont actives.
chain = {(0, 1), (1, 2), (2, 3)}
availability = {key: (key in chain) for key in instance_chain.residual_edges}

dv = DynamicStateValidator(instance_chain)
print("On veut couper (1, 2) :",
      dv.can_disable_edge(availability, (1, 2)))
print("On veut couper (0, 1) :",
      dv.can_disable_edge(availability, (0, 1)))

## 5. Resume mental

- `InstanceValidator` protege la **qualite statique** du graphe genere : densite raisonnable, degre moyen suffisant, connexite residuelle.
- `DynamicStateValidator` protege la **stabilite temporelle** de la simulation : pas de partition du reseau, pas de vidage progressif, pas de scenario ou trop d'aretes sont simultanement OFF.
- Les deux lisent les seuils `resolved_*` de `GraphGenerationConfig`, ce qui garde toutes les bornes au meme endroit.

A retenir : le generateur et le simulateur proposent, les validateurs disposent. C'est ce qui permet d'utiliser du hasard sans laisser le hasard casser les invariants du projet.

---

## References

[1] **Bondy, J. A. & Murty, U. S. R. (2008).** *Graph Theory.* Springer. https://doi.org/10.1007/978-1-84628-970-5

[2] **Cormen, T. H., Leiserson, C. E., Rivest, R. L. & Stein, C. (2022).** *Introduction to Algorithms* (4th ed.). MIT Press.

[3] **Diestel, R. (2017).** *Graph Theory* (5th ed.). Springer. https://doi.org/10.1007/978-3-662-53622-3
